# Day 4: Performance Analytics & Mutual Fund Scorecard

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import linregress

processed_dir = Path("../data/processed")
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# Load cleaned datasets
fm = pd.read_csv(processed_dir / "clean_fund_master.csv")
nav = pd.read_csv(processed_dir / "clean_nav.csv")
bench = pd.read_csv(processed_dir / "clean_benchmark_indices.csv")

nav['date'] = pd.to_datetime(nav['date'])
bench['date'] = pd.to_datetime(bench['date'])

## 1. Setting up NIFTY100 Daily Returns

In [2]:
nifty100 = bench[bench['index_name'] == 'NIFTY100'].sort_values('date').copy()
nifty100['nifty_return'] = nifty100['close_value'].pct_change()
nifty100_clean = nifty100.dropna(subset=['nifty_return'])
print(f"Benchmark entries parsed: {nifty100_clean.shape[0]}")

Benchmark entries parsed: 1149


## 2. Core Metrics Calculation

In [3]:
rf = 0.065
daily_rf = rf / 252
results = []

for code, group in nav.groupby('amfi_code'):
    group = group.sort_values('date')
    group['daily_return'] = group['nav'].pct_change()
    latest_date = group['date'].max()
    
    # 1. CAGR Calculation
    cagrs = {}
    for yrs in [1, 3, 5]:
        start_date = latest_date - pd.DateOffset(years=yrs)
        past = group[group['date'] <= start_date]
        if not past.empty:
            start_row = past.iloc[-1]
            end_row = group.iloc[-1]
            n_trading = len(group[(group['date'] >= start_row['date']) & (group['date'] <= end_row['date'])])
            cagrs[yrs] = (end_row['nav'] / start_row['nav']) ** (252 / n_trading) - 1
        else:
            cagrs[yrs] = np.nan
            
    clean_ret = group['daily_return'].dropna()
    mean_ret = clean_ret.mean()
    std_ret = clean_ret.std()
    
    # 2. Sharpe Ratio
    sharpe = (mean_ret - daily_rf) / std_ret * np.sqrt(252) if std_ret > 0 else np.nan
    
    # 3. Sortino Ratio
    downside_ret = clean_ret[clean_ret < 0]
    downside_std = downside_ret.std()
    sortino = (mean_ret - daily_rf) / downside_std * np.sqrt(252) if downside_std > 0 else np.nan
    
    # 4. Alpha & Beta (CAPM)
    merged = pd.merge(group[['date', 'daily_return']], nifty100_clean[['date', 'nifty_return']], on='date').dropna()
    if len(merged) > 30:
        slope, intercept, r_val, p_val, std_err = linregress(merged['nifty_return'], merged['daily_return'])
        beta = slope
        alpha = intercept * 252
    else:
        beta, alpha = np.nan, np.nan
        
    # 5. Maximum Drawdown
    running_max = group['nav'].cummax()
    drawdown = group['nav'] / running_max - 1
    max_dd = drawdown.min()
    
    end_idx = drawdown.idxmin()
    end_date = group.loc[end_idx, 'date']
    peak_idx = group[group['date'] <= end_date]['nav'].idxmax()
    start_date_dd = group.loc[peak_idx, 'date']
    
    fund_info = fm[fm['amfi_code'] == code].iloc[0]
    
    results.append({
        'amfi_code': code,
        'scheme_name': fund_info['scheme_name'],
        'fund_house': fund_info['fund_house'],
        'expense_ratio_pct': fund_info['expense_ratio_pct'],
        'cagr_1yr': cagrs[1],
        'cagr_3yr': cagrs[3],
        'cagr_5yr': cagrs[5],
        'sharpe': sharpe,
        'sortino': sortino,
        'alpha': alpha,
        'beta': beta,
        'max_dd': max_dd,
        'max_dd_start': start_date_dd.strftime('%Y-%m-%d'),
        'max_dd_end': end_date.strftime('%Y-%m-%d')
    })

res_df = pd.DataFrame(results)
print(f"Calculation complete for {len(res_df)} funds.")

Calculation complete for 40 funds.


## 3. Scorecard Design & Ranking

In [4]:
res_df['rank_3yr'] = res_df['cagr_3yr'].rank(pct=True) * 100
res_df['rank_sharpe'] = res_df['sharpe'].rank(pct=True) * 100
res_df['rank_alpha'] = res_df['alpha'].rank(pct=True) * 100
res_df['rank_expense'] = res_df['expense_ratio_pct'].rank(ascending=False, pct=True) * 100
res_df['rank_max_dd'] = res_df['max_dd'].rank(ascending=False, pct=True) * 100

res_df['score'] = (
    0.30 * res_df['rank_3yr'] +
    0.25 * res_df['rank_sharpe'] +
    0.20 * res_df['rank_alpha'] +
    0.15 * res_df['rank_expense'] +
    0.10 * res_df['rank_max_dd']
)

scorecard_cols = ['amfi_code', 'scheme_name', 'fund_house', 'cagr_1yr', 'cagr_3yr', 'cagr_5yr', 'sharpe', 'sortino', 'max_dd', 'score']
scorecard = res_df[scorecard_cols].sort_values(by='score', ascending=False)
scorecard.to_csv(reports_dir / "fund_scorecard.csv", index=False)
res_df[['amfi_code', 'scheme_name', 'alpha', 'beta']].to_csv(reports_dir / "alpha_beta.csv", index=False)

# Render formatted and color-coded table
scorecard.style.background_gradient(subset=['cagr_3yr', 'sharpe', 'score'], cmap='RdYlGn')\
               .format({
                   'cagr_1yr': '{:.2%}',
                   'cagr_3yr': '{:.2%}',
                   'cagr_5yr': '{:.2%}',
                   'sharpe': '{:.3f}',
                   'sortino': '{:.3f}',
                   'max_dd': '{:.2%}',
                   'score': '{:.1f}'
               })

,amfi_code,scheme_name,fund_house,cagr_1yr,cagr_3yr,cagr_5yr,sharpe,sortino,max_dd,score
25,120505,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,19.55%,20.95%,nan%,0.883,1.286,-18.19%,85.1
16,119094,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,14.84%,23.05%,nan%,0.731,1.055,-20.96%,82.0
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Growth,HDFC Mutual Fund,34.16%,21.37%,nan%,0.808,1.144,-16.22%,80.5
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,13.61%,22.35%,nan%,1.068,1.491,-11.27%,80.0
30,120843,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,17.67%,19.55%,nan%,0.966,1.480,-12.97%,78.2
21,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,51.47%,17.69%,nan%,0.712,1.067,-28.71%,75.8
39,149324,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,41.25%,17.91%,nan%,0.714,1.031,-31.17%,75.6
24,120504,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,8.82%,21.39%,nan%,0.715,1.064,-12.59%,75.1
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset MF,25.92%,19.30%,nan%,0.919,1.353,-16.40%,73.9
19,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,38.47%,20.11%,nan%,0.861,1.291,-15.01%,72.4
